In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device

device(type='cuda')

In [ ]:
class Mymodel(nn.Module):
    def __init__(self, dim_list, activation_list):
        super().__init__()
        self.num_layers = len(dim_list) - 1
        self.layers = nn.Sequential()

        for l in range(self.num_layers):
            self.layers.append(nn.ModuleList())
            self.layers[-1].append(nn.Linear(dim_list[l], dim_list[l + 1]))
            self.layers[-1].append(getattr(nn, activation_list[l])())

            # If the activation function is not ReLU or GELU, then for the
            # linear module, we do Xavier's uniform initialization
            # and all zero initialization on bias (if it is not None)
            if activation_list[l] not in ['ReLU', 'GELU']:
                init.xavier_uniform_(self.layers[l][0].weight)
                if self.layers[l][0].bias is not None:
                    init.zeros_(self.layers[l][0].bias)

    def forward(self, x):
        Y = x
        for l in range(1, self.num_layers + 1):
            Z = self.layers[l - 1][0](Y)
            Y = self.layers[l - 1][1](Z)

        return Y

In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784')
inputs, outputs = mnist["data"], mnist["target"]

inputs = np.array(inputs)
outputs = np.array(outputs.astype(int))

# We do experiment by using a subset of data only (to save time)
sample_size = 2000
np.random.seed(42)
indices = np.random.choice(inputs.shape[0], sample_size, replace=False)
np.random.seed()

inputs = inputs[indices]
outputs = outputs[indices]

# Normalize inputs to [0, 1]
inputs = inputs / 255

In [ ]:
inputs = torch.tensor(inputs, dtype=torch.float32)
outputs = torch.tensor(outputs, dtype=torch.long)

In [ ]:
random_state = 42
inputs_train, inputs_test, outputs_train, outputs_test = train_test_split(inputs, outputs, test_size=0.2, random_state=random_state)

In [ ]:
class MyDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __len__(self):
        return self.inputs.shape[0]

    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

dataset_train = MyDataset(inputs_train, outputs_train)
dataset_test = MyDataset(inputs_test, outputs_test)

In [ ]:
batch_size = 32
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=len(dataset_test), shuffle=True)

In [ ]:
dim_list = [784, 64, 10]
activation_list =['ReLU', 'Identity']
model_MNIST = Mymodel(dim_list, activation_list)

In [ ]:
lr = 1e-4
optimizer  = torch.optim.Adam(model_MNIST.parameters(), lr=lr)

In [ ]:

criterion = nn.CrossEntropyLoss()

In [ ]:
num_epochs = 100
loss_list = []
f1_list = []
for n in tqdm(range(num_epochs)):
    model_MNIST.train()
    for inputs, outputs in dataloader_train:
        inputs = inputs.to(device)
        outputs = outputs.to(device)
        model_MNIST.to(device)

        optimizer.zero_grad()
        pred_logits = model_MNIST(inputs)
        loss = criterion(pred_logits, outputs)

        loss.backward()
        optimizer.step()

        model_MNIST.eval()

        model_MNIST.to('cpu')
        with torch.no_grad():
            logits = model_MNIST(inputs_train)

            pred_prob = nn.Softmax(dim=1)(logits)
            loss_list.append(criterion(logits, outputs_train).item())

            pred_label = torch.argmax(pred_prob, dim=1)
            f1_list.append(f1_score(outputs_train, pred_label, average='macro'))

In [ ]:

plt.plot(loss_list)
plt.title("Loss function")
plt.show()

plt.plot(f1_list)
plt.title("Prediction f1 score")
plt.show()

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

bank_marketing = fetch_ucirepo(id=222)
X = bank_marketing.data.features
y = bank_marketing.data.targets

In [ ]:
print(bank_marketing.metadata)
print(bank_marketing.variables)
print(X.head(10))

In [ ]:
print(y.shape)

In [ ]:
X = X.drop(['contact', 'poutcome'], axis=1)

missing_rows = X.isnull().any(axis = 1)

X = X.dropna()

y = y.drop(missing_rows.index[missing_rows])

y = y.replace({'no': 0, 'yes': 1})

